# 01 · Preprocess

Load the two essays and the evidence cards, inspect them, and save the prepared LangChain documents to `data/processed/documents.jsonl` (used by the single-question demo).

No API key is needed for this notebook.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

In [ ]:
import json

import pandas as pd

from src.config import DOCUMENTS_FILE, ESSAY_DIR, ESSAYS, EVIDENCE_FILE
from src.data_loader import load_cards, load_essay, save_documents

## Essays

In [ ]:
essays = [load_essay(ESSAY_DIR / fname, name) for fname, name in ESSAYS]

for doc in essays:
    text = doc.page_content
    print(f"{doc.metadata['source']}: {len(text):,} characters, {len(text.split()):,} words")
    print(text[:300], "...\n")

## Evidence cards

In [ ]:
cards = load_cards(EVIDENCE_FILE)
raw_cards = json.loads(EVIDENCE_FILE.read_text(encoding="utf-8"))

pd.DataFrame(
    [{"title": c["title"], "year": c["year"], "type": c["type"], "factors": len(c["factors"])} for c in raw_cards]
)

In [ ]:
# One row per risk factor across all cards
factors = pd.DataFrame(
    [
        {
            "card": c["title"][:45] + "...",
            "factor": f["factor"],
            "category": f["category"],
            "effect": f["effect_direction"],
            "magnitude": f["magnitude"],
        }
        for c in raw_cards
        for f in c["factors"]
    ]
)
factors

## Save prepared documents

In [ ]:
docs = essays + cards
save_documents(docs, DOCUMENTS_FILE)
print(f"Saved {len(docs)} documents to {DOCUMENTS_FILE.relative_to(ROOT)}")